# 01 — Network and route inputs

This notebook checks the geographic inputs behind the map outputs. It deliberately does **not** download the OpenStreetMap road network unless you explicitly run the optional final cell.

## Questions this notebook answers

- Are route geometries present and readable?
- Which route attributes are available for maps?
- Do route assumptions have the same order as the model routes?

The game calculations use the reviewed assumptions in `src/config.py`; route geometry is used for mapping and spatial checks.

In [ ]:
from pathlib import Path
import os

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)

from src.config import DEFAULT_ROUTE_INFO, ROUTES
from src.network import build_road_graph, load_routes
from src.paths import ProjectPaths

paths = ProjectPaths.discover()
paths.route_geometries

In [ ]:
routes = load_routes(paths.route_geometries)
print(f'{len(routes)} route geometries loaded')
routes[['route_code', 'route_name', 'geometry']].head()

In [ ]:
assert not routes.empty, 'No route geometries were found.'
assert routes.geometry.notna().all(), 'At least one route has no geometry.'
assert routes.geometry.is_valid.all(), 'At least one route geometry is invalid.'

assumptions = {
    'route': ROUTES,
    'length_km': DEFAULT_ROUTE_INFO['length_km'],
    'cycle_time_min': DEFAULT_ROUTE_INFO['cycle_time_min'],
    'served_node_indices': DEFAULT_ROUTE_INFO['served_nodes'],
}
import pandas as pd
pd.DataFrame(assumptions)

In [ ]:
ax = routes.plot(figsize=(9, 8), linewidth=2, legend=True)
ax.set_title('Legazpi–Daraga route geometries')
ax.set_axis_off()

## Optional: build the road graph

This operation may download OpenStreetMap data the first time and can take several minutes. Run it only when you need the road-network map. The graph is cached in `cache/` afterward.

In [ ]:
# Uncomment only when a road-network map is required.
# graph = build_road_graph()
# print(f'Road graph: {len(graph.nodes):,} nodes, {len(graph.edges):,} edges')